**This script looks into group characterisation of consistency/reproducibility of ERPAC**

In [57]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import ttest_ind, t
from mne.stats import permutation_cluster_test

import os
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))
from config.paths import ERPAC_DIR
from config.config import COUPLINGS, TASK_STAGES, ROI, TOI, GROUPS

%matplotlib qt

A. Subject-to-group similarity (leave-one-out group mean map)
B. Pairwise inter-subject similarity (homogenuity and outliers)
C. Peak-feature reproducibility (peak ERPAC magnitude, latency amd gamma frequency)


1. ERPAC time-frequency maps

In [3]:
# ============================================================
# 1. LOAD TIME-RESOLVED ERPAC DATA
# ============================================================

erpac_df = pd.read_parquet(
    os.path.join(
        ERPAC_DIR,
        "erpac_results.parquet"
    )
)

erpac_df_toi = erpac_df[
    (
        (erpac_df["task_stage"] == "plan") &
        erpac_df["time"].between(
            TOI["plan"]["start"],
            TOI["plan"]["end"]
        )
    )
    |
    (
        (erpac_df["task_stage"] == "go") &
        erpac_df["time"].between(
            TOI["go"]["start"],
            TOI["go"]["end"]
        )
    )
].copy()

erpac_df_toi

,sub,group,task,task_stage,coupling,roi,amp_freq,time,erpac_value
250,s1_pac_sub01,Y,FTT,plan,theta_gamma,M1,32.5,0.000,0.106802
251,s1_pac_sub01,Y,FTT,plan,theta_gamma,M1,32.5,0.002,0.107298
252,s1_pac_sub01,Y,FTT,plan,theta_gamma,M1,32.5,0.004,0.108432
253,s1_pac_sub01,Y,FTT,plan,theta_gamma,M1,32.5,0.006,0.109629
254,s1_pac_sub01,Y,FTT,plan,theta_gamma,M1,32.5,0.008,0.110709
...,...,...,...,...,...,...,...,...,...
25430755,s1_pac_sub68,O,FTT,go,beta_gamma,SMA,76.5,0.492,0.113945
25430756,s1_pac_sub68,O,FTT,go,beta_gamma,SMA,76.5,0.494,0.109723
25430757,s1_pac_sub68,O,FTT,go,beta_gamma,SMA,76.5,0.496,0.105073
25430758,s1_pac_sub68,O,FTT,go,beta_gamma,SMA,76.5,0.498,0.104456


In [6]:
# Build a subject × freq × time array

def get_group_erpac_array(
    erpac_df,
    group,
    coupling,
    stage,
    roi,
    task="FTT",
):
    """
    Returns:
        data : ndarray, shape (n_subs, n_freqs, n_times)
        subs : list of subject IDs
        freqs : ndarray
        times : ndarray
    """

    df = erpac_df[
        (erpac_df["group"] == group) &
        (erpac_df["task"] == task) &
        (erpac_df["coupling"] == coupling) &
        (erpac_df["task_stage"] == stage) &
        (erpac_df["roi"] == roi)
    ].copy()

    freqs = np.sort(df["amp_freq"].unique())
    times = np.sort(df["time"].unique())
    subs = sorted(df["sub"].unique())

    data = np.full(
        (len(subs), len(freqs), len(times)),
        np.nan
    )

    for i, sub in enumerate(subs):

        sub_df = df[df["sub"] == sub]

        pivot = sub_df.pivot(
            index="amp_freq",
            columns="time",
            values="erpac_value"
        )

        pivot = pivot.reindex(
            index=freqs,
            columns=times
        )

        data[i] = pivot.to_numpy()

    return data, subs, freqs, times

In [7]:
# Subject-to-group similarity

def flatten_map(x):
    return x.reshape(-1)


def zscore_map(x):
    x = flatten_map(x)
    return (x - np.nanmean(x)) / np.nanstd(x)


def map_correlation(a, b, zscore=False):
    if zscore:
        a = zscore_map(a)
        b = zscore_map(b)
    else:
        a = flatten_map(a)
        b = flatten_map(b)

    mask = ~np.isnan(a) & ~np.isnan(b)

    if mask.sum() < 3:
        return np.nan

    return np.corrcoef(a[mask], b[mask])[0, 1]


def leave_one_out_similarity(data, zscore=False):
    """
    data: shape (n_subs, n_freqs, n_times)
    """
    n_subs = data.shape[0]
    corrs = []

    for i in range(n_subs):
        subj_map = data[i]
        group_mean = np.nanmean(
            np.delete(data, i, axis=0),
            axis=0
        )

        r = map_correlation(
            subj_map,
            group_mean,
            zscore=zscore
        )

        corrs.append(r)

    return np.array(corrs)

In [ ]:
# Pairwise inter-subject similarity

def pairwise_similarity_matrix(data, zscore=False):
    n_subs = data.shape[0]
    sim = np.full((n_subs, n_subs), np.nan)

    for i in range(n_subs):
        for j in range(n_subs):
            sim[i, j] = map_correlation(
                data[i],
                data[j],
                zscore=zscore
            )

    return sim

def mean_upper_triangle(mat):
    iu = np.triu_indices_from(mat, k=1)
    return np.nanmean(mat[iu])



In [ ]:
# Split-half reproducibility

def split_half_reliability(
    data,
    n_splits=1000,
    zscore=False,
    random_state=42,
):
    rng = np.random.default_rng(random_state)
    n_subs = data.shape[0]
    corrs = []

    for _ in range(n_splits):
        perm = rng.permutation(n_subs)
        half = n_subs // 2

        idx1 = perm[:half]
        idx2 = perm[half:]

        map1 = np.nanmean(data[idx1], axis=0)
        map2 = np.nanmean(data[idx2], axis=0)

        r = map_correlation(
            map1,
            map2,
            zscore=zscore
        )
        corrs.append(r)

    return np.array(corrs)

In [10]:
# Peak features

def extract_peak_features(data, freqs, times):
    rows = []

    for i in range(data.shape[0]):
        subj_map = data[i]

        if np.all(np.isnan(subj_map)):
            rows.append({
                "peak_value": np.nan,
                "peak_freq": np.nan,
                "peak_time": np.nan,
            })
            continue

        flat_idx = np.nanargmax(subj_map)
        f_idx, t_idx = np.unravel_index(
            flat_idx,
            subj_map.shape
        )

        rows.append({
            "peak_value": subj_map[f_idx, t_idx],
            "peak_freq": freqs[f_idx],
            "peak_time": times[t_idx],
        })

    return pd.DataFrame(rows)


In [11]:
# WRAPPER 

def characterize_group_condition(
    erpac_df,
    group,
    coupling,
    stage,
    roi,
    task="FTT",
):
    data, subs, freqs, times = get_group_erpac_array(
        erpac_df=erpac_df,
        group=group,
        coupling=coupling,
        stage=stage,
        roi=roi,
        task=task,
    )

    group_mean = np.nanmean(data, axis=0)

    loo_raw = leave_one_out_similarity(
        data,
        zscore=False
    )

    loo_shape = leave_one_out_similarity(
        data,
        zscore=True
    )

    pairwise = pairwise_similarity_matrix(
        data,
        zscore=True
    )

    split_half = split_half_reliability(
        data,
        n_splits=1000,
        zscore=True
    )

    peak_df = extract_peak_features(
        data,
        freqs,
        times
    )
    peak_df["sub"] = subs

    summary = {
        "coupling": coupling,
        "stage": stage,
        "roi": roi,
        "n_subs": data.shape[0],
        "mean_loo_raw_r": np.nanmean(loo_raw),
        "mean_loo_shape_r": np.nanmean(loo_shape),
        "mean_pairwise_shape_r": mean_upper_triangle(pairwise),
        "median_split_half_r": np.nanmedian(split_half),
        "split_half_r_2p5": np.nanpercentile(split_half, 2.5),
        "split_half_r_97p5": np.nanpercentile(split_half, 97.5),
    }

    return {
        "summary": summary,
        "data": data,
        "subs": subs,
        "freqs": freqs,
        "times": times,
        "group_mean": group_mean,
        "loo_raw": loo_raw,
        "loo_shape": loo_shape,
        "pairwise": pairwise,
        "split_half": split_half,
        "peak_df": peak_df,
    }

**PLOTTING**

In [31]:
def plot_loo_similarity(
    result,
    group,
    coupling,
    stage,
    roi,
):

    correlations = result["loo_shape"]
    subs = result["subs"]

    fig, ax = plt.subplots(
        figsize=(7, 4.5)
    )

    # Individual participants
    x = np.arange(len(correlations))

    ax.scatter(
        x,
        correlations,
        s=45,
        alpha=0.75,
    )

    # Mean
    mean_r = np.nanmean(correlations)

    ax.axhline(
        mean_r,
        linestyle="--",
        linewidth=1.5,
        label=f"Mean r = {mean_r:.2f}",
    )

    # Zero correlation
    ax.axhline(
        0,
        linewidth=0.8,
        alpha=0.4,
    )

    ax.set_xticks(x)
    ax.set_xticklabels(
        subs,
        rotation=60,
        ha="right",
        fontsize=8,
    )

    ax.set_ylabel(
        "Leave-one-out map correlation (r)"
    )

    ax.set_xlabel(
        "Participant"
    )

    ax.set_title(
        f"{group} | {coupling} | {stage} | {roi}",
        fontweight="bold",
    )

    ax.legend(
        frameon=False
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()

    return fig


In [32]:
def plot_pairwise_similarity(
    result,
    group,
    coupling,
    stage,
    roi,
):

    similarity = result["pairwise"]
    subs = result["subs"]

    fig, ax = plt.subplots(
        figsize=(8, 7)
    )

    im = ax.imshow(
        similarity,
        vmin=-1,
        vmax=1,
        aspect="equal",
    )

    cbar = fig.colorbar(
        im,
        ax=ax,
        shrink=0.8,
    )

    cbar.set_label(
        "ERPAC map correlation (r)"
    )

    ax.set_xticks(
        np.arange(len(subs))
    )

    ax.set_yticks(
        np.arange(len(subs))
    )

    ax.set_xticklabels(
        subs,
        rotation=90,
        fontsize=7,
    )

    ax.set_yticklabels(
        subs,
        fontsize=7,
    )

    ax.set_xlabel(
        "Participant"
    )

    ax.set_ylabel(
        "Participant"
    )

    ax.set_title(
        f"Inter-participant ERPAC similarity\n"
        f"{group} | {coupling} | {stage} | {roi}",
        fontweight="bold",
    )

    plt.tight_layout()

    return fig


In [33]:
def plot_split_half(
    result,
    group,
    coupling,
    stage,
    roi,
):

    correlations = result["split_half"]

    median_r = np.nanmedian(
        correlations
    )

    lower = np.nanpercentile(
        correlations,
        2.5
    )

    upper = np.nanpercentile(
        correlations,
        97.5
    )

    fig, ax = plt.subplots(
        figsize=(7, 4.5)
    )

    ax.hist(
        correlations,
        bins=30,
        alpha=0.75,
        edgecolor="white",
    )

    # Median
    ax.axvline(
        median_r,
        linestyle="--",
        linewidth=2,
        label=f"Median r = {median_r:.2f}",
    )

    # 95% interval
    ax.axvline(
        lower,
        linestyle=":",
        linewidth=1.5,
    )

    ax.axvline(
        upper,
        linestyle=":",
        linewidth=1.5,
    )

    ax.set_xlabel(
        "Correlation between split-half mean maps (r)"
    )

    ax.set_ylabel(
        "Number of splits"
    )

    ax.set_title(
        f"Split-half reproducibility\n"
        f"{group} | {coupling} | {stage} | {roi}",
        fontweight="bold",
    )

    ax.legend(
        frameon=False
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()

    return fig


In [34]:
def plot_peak_features(
    result,
    group,
    coupling,
    stage,
    roi,
):

    peak_df = result["peak_df"].copy()

    fig, axes = plt.subplots(
        nrows=1,
        ncols=3,
        figsize=(11, 4),
    )

    # ========================================================
    # Peak magnitude
    # ========================================================

    axes[0].scatter(
        np.ones(len(peak_df)),
        peak_df["peak_value"],
        alpha=0.7,
        s=45,
    )

    axes[0].axhline(
        peak_df["peak_value"].mean(),
        linestyle="--",
        linewidth=1.5,
    )

    axes[0].set_xlim(
        0.7,
        1.3,
    )

    axes[0].set_xticks([])

    axes[0].set_ylabel(
        "Peak ERPAC"
    )

    axes[0].set_title(
        "Peak magnitude"
    )


    # ========================================================
    # Peak latency
    # ========================================================

    axes[1].scatter(
        np.ones(len(peak_df)),
        peak_df["peak_time"],
        alpha=0.7,
        s=45,
    )

    axes[1].axhline(
        peak_df["peak_time"].mean(),
        linestyle="--",
        linewidth=1.5,
    )

    axes[1].set_xlim(
        0.7,
        1.3,
    )

    axes[1].set_xticks([])

    axes[1].set_ylabel(
        "Peak latency (s)"
    )

    axes[1].set_title(
        "Peak timing"
    )


    # ========================================================
    # Peak gamma frequency
    # ========================================================

    axes[2].scatter(
        np.ones(len(peak_df)),
        peak_df["peak_freq"],
        alpha=0.7,
        s=45,
    )

    axes[2].axhline(
        peak_df["peak_freq"].mean(),
        linestyle="--",
        linewidth=1.5,
    )

    axes[2].set_xlim(
        0.7,
        1.3,
    )

    axes[2].set_xticks([])

    axes[2].set_ylabel(
        "Gamma frequency (Hz)"
    )

    axes[2].set_title(
        "Peak frequency"
    )


    # ========================================================
    # Formatting
    # ========================================================

    for ax in axes:

        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.spines["bottom"].set_visible(False)

    fig.suptitle(
        f"ERPAC peak characteristics\n"
        f"{group} | {coupling} | {stage} | {roi}",
        fontsize=14,
        fontweight="bold",
    )

    fig.tight_layout()

    return fig

One condition test

In [52]:
group = "O"
coupling = "theta_gamma"
task_stage = "go"
roi = "SMA"

result = characterize_group_condition(
    erpac_df=erpac_df_toi,
    group=group,
    coupling=coupling,
    stage=task_stage,
    roi=roi
)


C:\Users\a1902989\AppData\Local\Temp\ipykernel_30020\2634459905.py:9: RuntimeWarning: Mean of empty slice
  return (x - np.nanmean(x)) / np.nanstd(x)
d:\BonoKat\research project\motor_pac_aging\.venv\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1997: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


In [53]:
result

{'summary': {'coupling': 'theta_gamma',
  'stage': 'go',
  'roi': 'SMA',
  'n_subs': 24,
  'mean_loo_raw_r': np.float64(-0.016004999155553506),
  'mean_loo_shape_r': np.float64(-0.016004999155553517),
  'mean_pairwise_shape_r': np.float64(-0.004012232402477187),
  'median_split_half_r': np.float64(-0.030089837208094675),
  'split_half_r_2p5': np.float64(-0.134076693171844),
  'split_half_r_97p5': np.float64(0.07424095457375025)},
 'data': array([[[0.1381077 , 0.13852966, 0.1387516 , ..., 0.14935452,
          0.14824067, 0.14720646],
         [0.15163705, 0.15232082, 0.15287255, ..., 0.14644624,
          0.14500487, 0.13664795],
         [0.16142333, 0.16201724, 0.16264351, ..., 0.13440852,
          0.13217175, 0.12076578],
         ...,
         [0.12151222, 0.12515388, 0.12799966, ..., 0.16133691,
          0.15975657, 0.13758506],
         [0.12751575, 0.13078165, 0.13300033, ..., 0.16235004,
          0.1601299 , 0.13604869],
         [0.13257398, 0.13531174, 0.13689883, ..., 0.1

In [54]:
# Pairwise inter-subject similarity

pairwise = result["pairwise"]

upper = np.triu_indices_from(
    pairwise,
    k=1
)

mean_pairwise_r = np.nanmean(
    pairwise[upper]
)

print(
    f"Mean pairwise r = "
    f"{mean_pairwise_r:.3f}"
)

Mean pairwise r = -0.004


In [55]:
# Plotting

fig1 = plot_loo_similarity(
    result,
    group=group,
    coupling=coupling,
    stage=task_stage,
    roi="SMA",
)

plt.show()

fig2 = plot_pairwise_similarity(
    result,
    group=group,
    coupling=coupling,
    stage=task_stage,
    roi=roi,
)

plt.show()

fig3 = plot_split_half(
    result,
    group=group,
    coupling=coupling,
    stage=task_stage,
    roi=roi,
)

plt.show()

fig4 = plot_peak_features(
    result,
    group=group,
    coupling=coupling,
    stage=task_stage,
    roi=roi,
)

plt.show()

peak_summary = (
        result["peak_df"][
            [
                "peak_value",
                "peak_time",
                "peak_freq",
            ]
        ].describe())

print(peak_summary)


       peak_value  peak_time  peak_freq
count   23.000000  23.000000  23.000000
mean     0.191160   0.125043  57.413043
std      0.024092   0.186231  14.881083
min      0.164274  -0.150000  32.500000
25%      0.172245  -0.049000  47.000000
50%      0.189430   0.074000  60.500000
75%      0.200374   0.281000  70.000000
max      0.267528   0.436000  76.500000


In [56]:
# Saving figures and peak summary

# Output folder
save_dir = os.path.join(ERPAC_DIR, "sample_characteristics")
os.makedirs(save_dir, exist_ok=True)

fig1.savefig(
    os.path.join(save_dir, f"{group}_{coupling}_{task_stage}_{roi}_loo_similarity.png"),
    dpi=300,
    bbox_inches="tight"
)

fig2.savefig(
    os.path.join(save_dir, f"{group}_{coupling}_{task_stage}_{roi}_pairwise_similarity.png"),
    dpi=300,
    bbox_inches="tight"
)

fig3.savefig(
    os.path.join(save_dir, f"{group}_{coupling}_{task_stage}_{roi}_split_half.png"),
    dpi=300,
    bbox_inches="tight"
)

fig4.savefig(
    os.path.join(save_dir, f"{group}_{coupling}_{task_stage}_{roi}_peak_features.png"),
    dpi=300,
    bbox_inches="tight"
)

peak_summary.to_csv(
    os.path.join(
        save_dir,
        f"{group}_{coupling}_{task_stage}_{roi}_peak_summary.csv"
    )
)


ITERATE THROUGH GROUPS AND CONDITIONS

In [60]:
# SAMPLE CHARACTERISTICS FOR ALL GROUPS AND CONDITIONS
for group in GROUPS: # GROUPS
    for coupling in COUPLINGS: # COUPLINGS
        for task_stage in TASK_STAGES: # TASK_STAGES
            for roi in ROI:

                result = characterize_group_condition(
                    erpac_df=erpac_df_toi,
                    group=group,
                    coupling=coupling,
                    stage=task_stage,
                    roi=roi
                )

                # Saving figures and peak summary

                # Output folder
                save_dir = os.path.join(ERPAC_DIR, "sample_characteristics")

                fig1 = plot_loo_similarity(
                    result,
                    group=group,
                    coupling=coupling,
                    stage=task_stage,
                    roi="SMA",
                )

                plt.show()

                fig2 = plot_pairwise_similarity(
                    result,
                    group=group,
                    coupling=coupling,
                    stage=task_stage,
                    roi=roi,
                )

                plt.show()

                fig3 = plot_split_half(
                    result,
                    group=group,
                    coupling=coupling,
                    stage=task_stage,
                    roi=roi,
                )

                plt.show()

                fig4 = plot_peak_features(
                    result,
                    group=group,
                    coupling=coupling,
                    stage=task_stage,
                    roi=roi,
                )

                plt.show()

                peak_summary = (
                        result["peak_df"][
                            [
                                "peak_value",
                                "peak_time",
                                "peak_freq",
                            ]
                        ].describe())
                
                fig1.savefig(
                    os.path.join(save_dir, f"{group}_{coupling}_{task_stage}_{roi}_loo_similarity.png"),
                    dpi=300,
                    bbox_inches="tight"
                )

                fig2.savefig(
                    os.path.join(save_dir, f"{group}_{coupling}_{task_stage}_{roi}_pairwise_similarity.png"),
                    dpi=300,
                    bbox_inches="tight"
                )

                fig3.savefig(
                    os.path.join(save_dir, f"{group}_{coupling}_{task_stage}_{roi}_split_half.png"),
                    dpi=300,
                    bbox_inches="tight"
                )

                fig4.savefig(
                    os.path.join(save_dir, f"{group}_{coupling}_{task_stage}_{roi}_peak_features.png"),
                    dpi=300,
                    bbox_inches="tight"
                )

                peak_summary.to_csv(
                    os.path.join(
                        save_dir,
                        f"{group}_{coupling}_{task_stage}_{roi}_peak_summary.csv"
                    )
                )


C:\Users\a1902989\AppData\Local\Temp\ipykernel_30020\2634459905.py:9: RuntimeWarning: Mean of empty slice
  return (x - np.nanmean(x)) / np.nanstd(x)
d:\BonoKat\research project\motor_pac_aging\.venv\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1997: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
C:\Users\a1902989\AppData\Local\Temp\ipykernel_30020\2634459905.py:9: RuntimeWarning: Mean of empty slice
  return (x - np.nanmean(x)) / np.nanstd(x)
d:\BonoKat\research project\motor_pac_aging\.venv\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1997: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
C:\Users\a1902989\AppData\Local\Temp\ipykernel_30020\3283935818.py:12: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much 

GROUP DISTRIBUTIONS

In [ ]:
import seaborn as sns

# ============================================================
# 1. LOAD DATA
# ============================================================

df = pd.read_csv(
    os.path.join(
        ERPAC_DIR,
        "erpac_no_time.csv"
    )
)


# ============================================================
# 2. ORDER CATEGORIES
# ============================================================

group_order = ["Y", "O"]
roi_order = ["M1", "S1", "PMC", "SMA"]

couplings = [
    "theta_gamma",
    "alpha_gamma",
    "beta_gamma"
]

stages = [
    "plan",
    "go"
]


# ============================================================
# 3. PLOT INDIVIDUAL PARTICIPANTS BY ROI
# ============================================================

def plot_individual_erpac(
    df,
    coupling,
    stage,
):

    df_plot = df[
        (df["coupling"] == coupling)
        & (df["task_stage"] == stage)
    ].copy()

    fig, axes = plt.subplots(
        1,
        4,
        figsize=(14, 4),
        sharey=True,
    )

    for ax, roi in zip(axes, roi_order):

        roi_df = df_plot[
            df_plot["roi"] == roi
        ]

        # Boxplot
        sns.boxplot(
            data=roi_df,
            x="group",
            y="erpac_value",
            order=group_order,
            showfliers=False,
            ax=ax,
        )

        # Individual participant values
        sns.stripplot(
            data=roi_df,
            x="group",
            y="erpac_value",
            order=group_order,
            jitter=0.15,
            alpha=0.65,
            size=5,
            ax=ax,
        )

        # Mean
        means = (
            roi_df
            .groupby("group")["erpac_value"]
            .mean()
            .reindex(group_order)
        )

        ax.scatter(
            range(len(group_order)),
            means.values,
            marker="D",
            s=60,
            zorder=5,
        )

        ax.set_title(roi)
        ax.set_xlabel("")

        if ax == axes[0]:
            ax.set_ylabel("Mean ERPAC")
        else:
            ax.set_ylabel("")

        ax.set_xticklabels(
            ["Young", "Old"]
        )

    fig.suptitle(
        f"{coupling} | {stage}",
        fontsize=14,
        fontweight="bold",
    )

    plt.tight_layout()
    plt.show()

# ============================================================
# 4. AVERAGE ACROSS ROI
# ============================================================

df_global = (
    df
    .groupby(
        [
            "sub",
            "group",
            "task",
            "task_stage",
            "coupling",
        ],
        as_index=False,
    )["erpac_value"]
    .mean()
)

# ============================================================
# 5. GLOBAL GROUP PLOT
# ============================================================

fig, axes = plt.subplots(
    2,
    3,
    figsize=(12, 8),
    sharey=True,
)

for row, stage in enumerate(stages):

    for col, coupling in enumerate(couplings):

        ax = axes[row, col]

        df_plot = df_global[
            (df_global["task_stage"] == stage)
            & (df_global["coupling"] == coupling)
        ]

        # Boxplots
        sns.boxplot(
            data=df_plot,
            x="group",
            y="erpac_value",
            order=group_order,
            showfliers=False,
            ax=ax,
        )

        # Individual participants
        sns.stripplot(
            data=df_plot,
            x="group",
            y="erpac_value",
            order=group_order,
            jitter=0.15,
            size=5,
            alpha=0.65,
            ax=ax,
        )

        # Group means
        means = (
            df_plot
            .groupby("group")["erpac_value"]
            .mean()
            .reindex(group_order)
        )

        ax.scatter(
            range(len(group_order)),
            means.values,
            marker="D",
            s=70,
            zorder=5,
        )

        ax.set_xticklabels(
            ["Young", "Old"]
        )

        ax.set_xlabel("")

        if row == 0:
            ax.set_title(
                coupling.replace("_", "–")
            )

        if col == 0:
            ax.set_ylabel(
                f"{stage}\nMean ERPAC"
            )
        else:
            ax.set_ylabel("")

plt.tight_layout()
plt.show()

C:\Users\a1902989\AppData\Local\Temp\ipykernel_38724\1593139788.py:198: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(
C:\Users\a1902989\AppData\Local\Temp\ipykernel_38724\1593139788.py:198: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(
C:\Users\a1902989\AppData\Local\Temp\ipykernel_38724\1593139788.py:198: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(
C:\Users\a1902989\AppData\Local\Temp\ipykernel_38724\1593139788.py:198: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(
C:\Users\a1902989\AppData\Local\Temp\ipykernel_38724\1593139788.py:198: UserWarning: set_ticklabels() should onl

In [ ]:
# ============================================================
# 1. LOAD DATA
# ============================================================

df = pd.read_csv(
    os.path.join(
        ERPAC_DIR,
        "erpac_no_time.csv"
    )
)


# ============================================================
# 2. SETTINGS
# ============================================================

group_order = ["Y", "O"]
roi_order = ["M1", "S1", "PMC", "SMA"]

couplings = [
    "theta_gamma",
    "alpha_gamma",
    "beta_gamma"
]

stages = [
    "plan",
    "go"
]


group_labels = {
    "Y": "Young",
    "O": "Older"
}

coupling_labels = {
    "theta_gamma": r"$\theta$–$\gamma$",
    "alpha_gamma": r"$\alpha$–$\gamma$",
    "beta_gamma": r"$\beta$–$\gamma$"
}

stage_labels = {
    "plan": "Planning",
    "go": "Execution"
}


# Colour palette
palette = {
    "Y": "#4C78A8",
    "O": "#E07B53"
}


# ============================================================
# 3. GLOBAL STYLE
# ============================================================

sns.set_theme(
    style="whitegrid",
    context="paper"
)

plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
})


# ============================================================
# 4. PLOT INDIVIDUAL PARTICIPANTS BY ROI
# ============================================================

def plot_individual_erpac(
    df,
    coupling,
    stage,
):

    df_plot = df[
        (df["coupling"] == coupling)
        & (df["task_stage"] == stage)
    ].copy()


    # --------------------------------------------------------
    # Shared y-axis limits
    # --------------------------------------------------------

    y_min = df_plot["erpac_value"].min()
    y_max = df_plot["erpac_value"].max()

    padding = (y_max - y_min) * 0.08

    y_lim = (
        y_min - padding,
        y_max + padding
    )


    # --------------------------------------------------------
    # Figure
    # --------------------------------------------------------

    fig, axes = plt.subplots(
        nrows=1,
        ncols=4,
        figsize=(12, 4.2),
        sharey=True,
    )


    for ax, roi in zip(
        axes,
        roi_order,
    ):

        roi_df = df_plot[
            df_plot["roi"] == roi
        ].copy()


        # ----------------------------------------------------
        # Boxplot
        # ----------------------------------------------------

        sns.boxplot(
            data=roi_df,
            x="group",
            y="erpac_value",
            order=group_order,
            hue="group",
            palette=palette,
            width=0.48,
            linewidth=1.1,
            showfliers=False,
            legend=False,
            boxprops={
                "alpha": 0.25
            },
            whiskerprops={
                "linewidth": 1
            },
            capprops={
                "linewidth": 1
            },
            medianprops={
                "color": "black",
                "linewidth": 1.4
            },
            ax=ax,
        )


        # ----------------------------------------------------
        # Individual participants
        # ----------------------------------------------------

        sns.stripplot(
            data=roi_df,
            x="group",
            y="erpac_value",
            order=group_order,
            hue="group",
            palette=palette,
            jitter=0.13,
            dodge=False,
            size=4,
            alpha=0.65,
            edgecolor="white",
            linewidth=0.35,
            legend=False,
            ax=ax,
        )


        # ----------------------------------------------------
        # Group means
        # ----------------------------------------------------

        means = (
            roi_df
            .groupby(
                "group",
                observed=False
            )["erpac_value"]
            .mean()
            .reindex(group_order)
        )


        for x, group in enumerate(group_order):

            ax.scatter(
                x,
                means.loc[group],
                s=65,
                marker="D",
                facecolor=palette[group],
                edgecolor="black",
                linewidth=0.8,
                zorder=10,
            )


        # ----------------------------------------------------
        # Formatting
        # ----------------------------------------------------

        ax.set_title(
            roi,
            fontweight="bold",
            pad=8
        )

        ax.set_xlabel("")

        ax.set_xticks(
            [0, 1]
        )

        ax.set_xticklabels(
            [
                group_labels["Y"],
                group_labels["O"]
            ]
        )

        ax.set_ylim(
            y_lim
        )

        if ax is axes[0]:

            ax.set_ylabel(
                "Mean ERPAC"
            )

        else:

            ax.set_ylabel("")


        # Light horizontal grid only
        ax.grid(
            axis="y",
            alpha=0.2,
            linewidth=0.7
        )

        ax.grid(
            axis="x",
            visible=False
        )

        sns.despine(
            ax=ax,
            trim=True
        )


    fig.suptitle(
        f"{coupling_labels[coupling]} coupling — "
        f"{stage_labels[stage]}",
        fontsize=15,
        fontweight="bold",
        y=1.03
    )

    fig.tight_layout()

    return fig


# ============================================================
# 5. AVERAGE ACROSS ROI
# ============================================================

df_global = (
    df
    .groupby(
        [
            "sub",
            "group",
            "task",
            "task_stage",
            "coupling",
        ],
        as_index=False,
        observed=False,
    )["erpac_value"]
    .mean()
)


# ============================================================
# 6. GLOBAL GROUP PLOT
# ============================================================

def plot_global_erpac(df_global):

    # --------------------------------------------------------
    # Common y-axis
    # --------------------------------------------------------

    y_min = df_global["erpac_value"].min()
    y_max = df_global["erpac_value"].max()

    padding = (
        y_max - y_min
    ) * 0.08

    y_lim = (
        y_min - padding,
        y_max + padding
    )


    # --------------------------------------------------------
    # Figure
    # --------------------------------------------------------

    fig, axes = plt.subplots(
        nrows=2,
        ncols=3,
        figsize=(10.5, 7),
        sharex=True,
        sharey=True,
    )


    for row, stage in enumerate(stages):

        for col, coupling in enumerate(couplings):

            ax = axes[row, col]

            df_plot = df_global[
                (df_global["task_stage"] == stage)
                & (df_global["coupling"] == coupling)
            ].copy()


            # ------------------------------------------------
            # Boxplot
            # ------------------------------------------------

            sns.boxplot(
                data=df_plot,
                x="group",
                y="erpac_value",
                order=group_order,
                hue="group",
                palette=palette,
                width=0.48,
                linewidth=1.1,
                showfliers=False,
                legend=False,
                boxprops={
                    "alpha": 0.25
                },
                medianprops={
                    "color": "black",
                    "linewidth": 1.4
                },
                ax=ax,
            )


            # ------------------------------------------------
            # Individual subjects
            # ------------------------------------------------

            sns.stripplot(
                data=df_plot,
                x="group",
                y="erpac_value",
                order=group_order,
                hue="group",
                palette=palette,
                jitter=0.13,
                size=4,
                alpha=0.65,
                edgecolor="white",
                linewidth=0.35,
                legend=False,
                ax=ax,
            )


            # ------------------------------------------------
            # Means
            # ------------------------------------------------

            means = (
                df_plot
                .groupby(
                    "group",
                    observed=False
                )["erpac_value"]
                .mean()
                .reindex(group_order)
            )


            for x, group in enumerate(group_order):

                ax.scatter(
                    x,
                    means.loc[group],
                    s=70,
                    marker="D",
                    facecolor=palette[group],
                    edgecolor="black",
                    linewidth=0.8,
                    zorder=10,
                )


            # ------------------------------------------------
            # Column titles
            # ------------------------------------------------

            if row == 0:

                ax.set_title(
                    coupling_labels[coupling],
                    fontweight="bold",
                    pad=10
                )


            # ------------------------------------------------
            # X-axis
            # ------------------------------------------------

            ax.set_xlabel("")

            ax.set_xticks(
                [0, 1]
            )

            ax.set_xticklabels(
                [
                    "Young",
                    "Older"
                ]
            )


            # ------------------------------------------------
            # Y-axis
            # ------------------------------------------------

            ax.set_ylim(
                y_lim
            )

            if col == 0:

                ax.set_ylabel(
                    "Mean ERPAC"
                )

            else:

                ax.set_ylabel("")


            # ------------------------------------------------
            # Grid
            # ------------------------------------------------

            ax.grid(
                axis="y",
                alpha=0.18,
                linewidth=0.7
            )

            ax.grid(
                axis="x",
                visible=False
            )

            sns.despine(
                ax=ax,
                trim=True
            )


        # ----------------------------------------------------
        # Stage label on left side of row
        # ----------------------------------------------------

        fig.text(
            0.035,
            0.68 if row == 0 else 0.30,
            stage_labels[stage],
            rotation=90,
            va="center",
            ha="center",
            fontsize=13,
            fontweight="bold",
        )


    fig.suptitle(
        "ERPAC across age groups",
        fontsize=16,
        fontweight="bold",
        y=0.94
    )

    fig.subplots_adjust(
        left=0.14,
        right=0.98,
        top=0.84,
        bottom=0.08,
        hspace=0.32,
        wspace=0.18,
    )

    return fig

fig = plot_global_erpac(
    df_global
)

plt.show()

plt.savefig(
        os.path.join(
            ERPAC_DIR,
            "erpac_group_distributions.png"
        ),
        dpi=300,
        bbox_inches="tight"
    )

In [ ]:
plt.savefig(
        os.path.join(
            ERPAC_DIR,
            "erpac_group_distributions.png"
        ),
        dpi=300,
        bbox_inches="tight"
    )